In [ ]:
from dataclasses import dataclass
from pathlib import Path

import kaggle_benchmarks as kbench
from kaggle_benchmarks.content_types import images
import pandas as pd

import abc_selective_attention_utils as utils

In [ ]:
FAMILY = "selective attention"
ATTENTIONAL_BASIS = "feature sensitive"
MODALITY = "visual"
TASK_TYPE = "counting"
TASK_NAME = f"{FAMILY} - {ATTENTIONAL_BASIS} {MODALITY} {TASK_TYPE}"


In [ ]:
@dataclass
class CountAnswer:
    count: int

In [ ]:
DATASET_ROOT = utils.DEFAULT_DATASET_ROOT
TASK_PATH = DATASET_ROOT / "selective_attention/feature_sensitive/visual"
CSV_PATH = TASK_PATH / "feature_sensitive_visual_full.csv"

df = utils.load_csv(CSV_PATH)
print(CSV_PATH)
print(TASK_PATH)
print("rows:", len(df))
print("columns:", sorted(df.columns.tolist()))

In [ ]:
base_cols = [
    "seed",
    "family",
    "attentional_basis",
    "modality",
    "dimension",
    "variant",
    "num_items",
    "rule_variant",
]

optional_cols = [
    col
    for col in [
        "target_count",
        "same_color_wrong_shape_count",
        "same_shape_wrong_color_count",
        "same_color_shape_wrong_size_count",
        "unrelated_count",
        "spatial_density",
        "layout_regularity",
        "fixed_size",
        "active_area_scale",
        "width",
        "height",
        "margin",
        "min_gap",
        "jitter",
    ]
    if col in df.columns
]

required_cols = ["image_path", "count_prompt", "gold_count"]
missing_required_cols = [col for col in required_cols if col not in df.columns]
if missing_required_cols:
    raise ValueError(f"Dataset is missing required columns: {missing_required_cols}")

count_eval_df = df[base_cols + optional_cols + required_cols].copy()
groupings = [
    cols
    for cols in [
        ["dimension"],
        ["dimension", "variant"],
        ["rule_variant"],
        ["num_items"],
        ["target_count"] if "target_count" in count_eval_df.columns else None,
        ["spatial_density"] if "spatial_density" in count_eval_df.columns else None,
        ["layout_regularity"] if "layout_regularity" in count_eval_df.columns else None,
    ]
    if cols is not None and all(col in count_eval_df.columns for col in cols)
]

failure_cols = [
    "seed",
    "dimension",
    "variant",
    "rule_variant",
    "gold_count",
    "predicted_count",
    "failure_type",
    "image_path",
]

print("count_eval_df columns:", count_eval_df.columns.tolist())
print("groupings:", groupings)


In [ ]:
@kbench.task(store_task=False)
def single_feature_sensitive_visual_count(
    llm,
    seed,
    family,
    attentional_basis,
    modality,
    dimension,
    variant,
    num_items,
    rule_variant,
    image_path,
    count_prompt,
    gold_count,
    target_count=None,
    same_color_wrong_shape_count=None,
    same_shape_wrong_color_count=None,
    same_color_shape_wrong_size_count=None,
    unrelated_count=None,
    spatial_density=None,
    layout_regularity=None,
    fixed_size=None,
    active_area_scale=None,
    width=None,
    height=None,
    margin=None,
    min_gap=None,
    jitter=None,
) -> dict:
    gold = int(gold_count)

    pred = None
    error = None
    error_type = None

    try:
        resolved_image_path = TASK_PATH / image_path
        img = images.from_path(resolved_image_path)
        response = llm.prompt(count_prompt, image=img, schema=CountAnswer)
        pred = int(response.count)
    except Exception as exc:
        error_type, error = utils.classify_prompt_error(exc)

    is_correct = pred == gold
    has_error = error_type is not None
    is_failure = not is_correct
    failure_type = "ok" if is_correct else (error_type if has_error else "wrong_answer")

    kbench.assertions.assert_true(
        is_correct,
        expectation=f"Expected {gold}, got {pred}" + (f" | error_type={error_type} | error={error}" if error else ""),
    )

    result = {
        "task": "feature_sensitive_visual_count_v1",
        "model": llm.name,
        "seed": int(seed),
        "family": family,
        "attentional_basis": attentional_basis,
        "modality": modality,
        "dimension": dimension,
        "variant": variant,
        "task_type": "counting",
        "num_items": int(num_items),
        "rule_variant": rule_variant,
        "target_count": int(target_count) if target_count not in (None, "") else None,
        "spatial_density": spatial_density,
        "layout_regularity": layout_regularity,
        "fixed_size": fixed_size if fixed_size not in ("", None) else None,
        "gold_count": gold,
        "predicted_count": pred,
        "image_path": str(image_path),
        "is_correct": is_correct,
        "is_failure": is_failure,
        "has_error": has_error,
        "failure_type": failure_type,
    }

    if error_type is not None:
        result["error_type"] = error_type
    if error is not None:
        result["error"] = error
    return result


@kbench.task(name="selective attention - feature sensitive visual counting")
def feature_sensitive_visual_count_dataset(llm, df) -> tuple[float, float]:
    with kbench.client.enable_cache():
        runs = single_feature_sensitive_visual_count.evaluate(
            stop_condition=lambda runs: len(runs) == df.shape[0],
            max_attempts=1,
            retry_delay=15,
            llm=[llm],
            evaluation_data=df,
            n_jobs=8,
            timeout=120,
            remove_run_files=False,
        )

    result = utils.summarize_and_log_runs(TASK_NAME, llm.name, runs, groupings, failure_cols)
    return result


In [ ]:
run = feature_sensitive_visual_count_dataset.run(kbench.llm, count_eval_df)
print(run)

In [ ]:
%choose feature_sensitive_visual_count_dataset